# V2a-RSN 220119_F2_run11: Primary M0/M1 Evaluation

This notebook runs the outer-owned probability comparison directly from shared temporal windows. Optional states, communities, and manifolds are not required.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.


In [ ]:
import sys
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


In [ ]:
from omegaconf import OmegaConf

from effectome.workflows import (
    EvaluationConfig,
    paired_log_loss_delta,
    score_classification_records,
    score_regression_records,
)
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_primary_probability_evaluation,
    stage_artifact_path,
)

DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220119_F2_run11"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RUN_ID = "primary"
METHOD = "primary-evaluation"
FORCE = False

shared_run = make_run(OUTPUT_ROOT, "reference", "shared")
WINDOWS_PATH = stage_artifact_path(shared_run, "windows")
EVALUATION_CONFIG_PATH = PROJECT_ROOT / "conf" / "evaluation" / "default.yaml"
evaluation_data = OmegaConf.to_container(OmegaConf.load(EVALUATION_CONFIG_PATH), resolve=True)
if not isinstance(evaluation_data, dict):
    raise TypeError(f"Expected a mapping in {EVALUATION_CONFIG_PATH}")
evaluation_data["endpoint_name"] = "continuous"
evaluation_data["endpoint_type"] = "continuous"
evaluation_cfg = EvaluationConfig(**evaluation_data)

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title="V2a-RSN 220119_F2_run11 primary probability evaluation",
    expected_stages=["windows_input", "primary_evaluation"],
    force=FORCE,
)


In [ ]:
result = run_primary_probability_evaluation(
    run,
    windows_path=WINDOWS_PATH,
    evaluation_cfg=evaluation_cfg,
    force=FORCE,
)
records = result.predictions
score_records = (
    score_classification_records
    if evaluation_cfg.endpoint_type == "categorical"
    else score_regression_records
)
for model_name in ("M0_activity", "M1_effectome"):
    print(score_records(
        records,
        model_name=model_name,
        endpoint_name=evaluation_cfg.endpoint_name,
    ))
print(paired_log_loss_delta(
    records,
    baseline_model="M0_activity",
    candidate_model="M1_effectome",
    endpoint_name=evaluation_cfg.endpoint_name,
))
print_stage_status(run)
